# CLOSE_ORIGINAL solves all our problems about Cash Dividend Adjustments + allows accessing of PIT Price for PIT Scanning/backtesting.

### tldr;
Stock events are fine to be adjusted. 
Cash Dividends distorts the PIT Price, which we don't want to do. Two ways of dealing with this:
1. `close_original` has the PIT Price for Scanning/backtesting and the "cash dividend-adjusted" `close` can be used for cross-sectional performance-based backtesting. Plus we can treat Cash Dividends as being re-invested - just like some major fintech websites do (prefered option as easy to do this and it's more flexible).
2. Apply Cash Dividend to Portfolio NAV (so that a $5 cash dividend and Stock price actually "trading" down by $5 on the Dividend are not registered as a loss when backtesting based on Price).


## Adjusting for Splits and Dividends
#### You should treat cash dividends differently than splits, especially for the timeframes you are trading (intraday).
Massive support: "For your question on splits vs dividends -- companies may issue dividends as form of stocks or cash (a.k.a. stock dividends vs cash dividends). When a stocks dividend happens (as you see from our split endpoint with adjustment_type=stock_dividend), those events impact stock's price and should be documented as splits; The cash dividends on the other hand has no impact to stock's price, and thus they are offered in a separated endpoint -- the Dividend endpoint. In short, if you need all price-impactful corporate split actions, only use the Split endpoint."

The following events change the share count / price scale and therefore require historical price adjustment:
- stock splits
- reverse splits
- stock dividends
- similar share-distribution actions

Those belong in the splits endpoint:
- [Massive Splits Endpoint Docs](https://massive.com/docs/rest/stocks/corporate-actions/splits?utm_source=chatgpt.com)

Ordinary cash dividends do **not** mechanically rescale intraday prices.

Cash dividends are exposed separately here:
- [Massive Dividends Endpoint Docs](https://massive.com/docs/rest/stocks/corporate-actions/dividends?utm_source=chatgpt.com)
#### A cash dividend is a fundamental transfer of value out of the company, not a mathematical restructuring of the shares. Because you are backtesting **intraday** and **swing** strategies using **1-min data**, adjusting historical prices for cash dividends can actually ruin your data. 
Here is how this impacts your specific strategies and how professional quants solve it.
#### Why cash dividends are different
Suppose:
- stock closes at $100
- company pays $1 cash dividend

On ex-dividend date, the market may open near:
- $99

But:
- the exchange did NOT retroactively divide all prior prices by 1%
- the company did NOT change share count
- liquidity/trading mechanics are unchanged

This is economically different from a split.

Suppose:
- stock closes at $100
- overnight:
    - $5 special dividend issued

Next day opens:
- ~$95

If you dividend-adjust minute bars:
- yesterday’s close becomes ~$95 retroactively

Now:
- the overnight gap disappears

But in reality:
- traders experienced a real gap
- overnight risk was real
- options repriced
- stops triggered
- liquidity changed

# 5.1 Downloading adjustments
Adjustment factors are stored in the Polygon folder under <code>raw/adjustments/AAPL.csv</code>. 

**If a ticker is recycled, then this file has the adjustment factors for all the companies associated with the ticker.** 
(Dev: This is fine since even for recycled tickers, adjustments obviously apply to the date in question and hence, the adjustment matches whatever recycled ticker is active at that time.)

To get the adjustments, we simply use the <code>Stock Splits</code> and <code>Dividends</code> endpoint through the SDK. These are <code>list_splits</code> and <code>list_dividends</code>. 

Ex-dividend is the date when the investor does *not* get the dividend. If an investor held the stock before, he does. So on ex-dividend date the stock on average drops with the dividend. We need to add the dividends back to the stock price. Or subtract them from before ex-dividend. Most platforms do the backwards adjustments so I will also do that. The advantage is that the current price in the data is then unadjusted and thus equal to the actual market price.

The execution date of a split is when the stock has just been split before market open. So all prices before the split should be adjusted. If the split is 10-to-1, this means that 10 stocks have become 1. So all prices before the execution date must be x10. If the split is 1-to-5, this means 1 stock is split into 5 pieces. Then all prices before the split date have to be divided by 5.

The file with adjustments in the <code>adjustments</code> folder have the following columns: <code>['ticker', 'date', 'type', 'subtype', 'amount']</code> The date is the ex-dividend or execution date. Type is 'DIV' or 'SPLIT'. Subtype is 'CD', 'SD', 'LT', 'ST' for dividends and 'R' (reverse), 'N' (regular) for splits. Amount is the USD amount for dividends and a fraction for splits. A 10-to-1 reverse split is 10. A 1-to-5 split is 0.2.

~~Note: dividends are [already adjusted](https://polygon.io/knowledge-base/article/does-polygon-adjust-historic-dividends-for-splits) for splits. So we don't have to do this again.~~ This is not true! See below.


# How is this coded?
If file exists, add new adjustments.

The script uses a CLEAN_DOWNLOAD flag, which controls whether to perform a fresh download of all data or to update an existing dataset incrementally.

📥 Data Retrieval Process

Date Filtering: Before making an API call, the script checks if a ticker's listing end_date is before the START_DATE. If so, it skips that ticker to avoid unnecessary API calls for securities that were already delisted.

### ^^^ this is wrong if we are downloading data for backtesting, cos for backtesting we need to apply adjustments to even tickers that are currently delisted so we can backtest on these delisted tickers  too cos at some point historically, they were active

Data Standardization: The raw data from the API is transformed into a consistent schema. The code renames columns, converts date formats, and creates a unified type column ('DIV' for dividends, 'SPLIT' for splits) to make the final dataset uniform.

Calculating Split Factors: For splits, a "split factor" is calculated as split_from / split_to. This factor can then be used to adjust historical prices. For example, a 2-for-1 split would have a factor of 2 / 1 = 2.0, while a 1-for-5 reverse split would be 1 / 5 = 0.2. The script also automatically categorizes the split subtype as either 'R' for reverse or 'N' for a normal split.

🛠️ Data Processing and Deduplication

Combining Data: The dividend and split DataFrames for a given ticker are concatenated and then sorted by date.

Deduplication: The script includes a crucial deduplication step to prevent errors. It checks for and removes duplicate entries, first for dividends, then for splits. This safeguards against potential issues caused by ticker recycling or API quirks.

💾 Update Logic

The script elegantly handles updates. If a file for a ticker already exists, it reads the old data, concatenates it with the new data, and then saves the combined, deduplicated result back to the same file.

Tracking Progress: To facilitate incremental updates, the script writes the END_DATE of the current run to a _end_date.txt file. On subsequent runs with CLEAN_DOWNLOAD=False, the script reads this file to determine a new START_DATE, resuming the download from where it left off.

In [1]:
from massive.rest import RESTClient
from datetime import datetime, date, time, timedelta
from times import first_trading_date_after_equal, last_trading_date_before_equal
from tickers import get_tickers
import os
import pandas as pd
import numpy as np

from dotenv import load_dotenv 

# Now load variables and define constants
load_dotenv() # to load variables from .env

ACCESS_KEY_ID = os.environ.get("ACCESS_KEY_ID")
if not ACCESS_KEY_ID:
    raise ValueError("No API key found. Please set ACCESS_KEY_ID in your .env file.")

API_KEY = os.environ.get("MASSIVE_API_KEY")
if not API_KEY:
    raise ValueError("No API key found. Please set MASSIVE_API_KEY in your .env file.")

client = RESTClient(api_key=API_KEY)

POLYGON_DATA_PATH = "../data/polygon/"

START_DATE = date(2016, 6, 1) # only go as far as your polygon membership allows. don't waste loop cycles
END_DATE = date(2026, 6, 15) # till today only or else you will get "out of range" from our custom function and don't waste loop cycles
# END_DATE = this will need updating everyday so that we only fetch the new adjustments in an idempotent way

# CLEAN_DOWNLOAD flag controls whether to perform a fresh download of all data or to update an existing dataset incrementally.
CLEAN_DOWNLOAD = True # If False, only update existing data to the END_DATE. If True, remove all files in adjustments/.

# Cash Dividends are NOT adjusted for splits by Polygon! MUST be done by us. Verified below.
## It's crucial to adjust Dividends for Splits!
https://massive.com/knowledge-base/article/does-massive-adjust-historic-dividends-for-splits


Why this is crucial:

https://aistudio.google.com/app/prompts?state=%7B%22ids%22:%5B%221CmuSHfJT0xy4UGGdELP5RLbrclRjvgiR%22%5D,%22action%22:%22open%22,%22userId%22:%22104962803681652166866%22,%22resourceKeys%22:%7B%7D%7D&usp=sharing

Imagine a stock in 2015. 
*   **Actual 2015 Price:** $100
*   **Actual 2015 Dividend Paid:** $4 per share
*   **Actual Dividend Yield in 2015:** 4% ($4 / $100)

Now, in 2020, the company does a **4-for-1 stock split**. 
Fast forward to today. You are pulling historical data for a backtest. Because of the 4-for-1 split, your data provider adjusts the historical 2015 price down by a factor of 4 to stitch the time series together smoothly.
*   **Adjusted 2015 Price:** $25 ($100 / 4)

**Here is where the problem happens if you DON'T adjust the dividend:**
If you take the *Adjusted 2015 Price* ($25) but use the *Unadjusted 2015 Dividend* ($4), your backtesting engine will calculate the historical dividend yield as:
$4 / $25 = **16% Dividend Yield.**

Suddenly, your backtest thinks this stock was yielding 16% in 2015. Your algorithmic value-factor strategy goes crazy, buys a massive simulated position, and shows you incredible backtested returns. In reality, the yield was only 4%. You've just introduced a massive data artifact.

To fix this, the data provider **adjusts the historical dividend for the split** (dividing it by 4).
*   **Adjusted 2015 Dividend:** $1 ($4 / 4)
*   **Correct Yield Calculation:** $1 / $25 = **4%**. 

Math is preserved. Reality is preserved.

### Verifying whether Dividends are adjusted for Split already or not.
The column likely_adjusted will be True if the actual ratio is closer to 1.0, False if it is closer to the split factor.

If you see several splits where the ratio is near the split factor, Polygon.io is not adjusting historic dividends for splits.

If the ratio is consistently near 1.0, then the notebook’s original assumption is correct and the data is already split‑adjusted.

Run this on a handful of liquid stocks with known splits; the pattern should be clear almost immediately.

**This fetches both dividends and splits for a ticker, then checks whether the pre‑split dividend amounts have been scaled by the split factor.**

In [2]:
# Helper functions to fetch data from Massive
def get_dividends(ticker: str, limit: int = 500) -> pd.DataFrame:
    """
    Fetch dividend history from Massive.
    Massive’s /v2/dividends returns a list of dicts.
    We normalise into a DataFrame with datetime columns.
    """
    try:
        data = client.list_dividends(ticker=ticker, limit=limit)  # returns list of dicts
    except AttributeError:
        # Fallback to raw request if the high‑level method is not available
        resp = client.get(f"/v2/dividends/{ticker}", params={"limit": limit})
        data = resp.json()
        if isinstance(data, dict):
            data = data.get("results", [])

    df = pd.DataFrame(data)
    if df.empty:
        return df

    # Normalise column names (Massive uses snake_case; dates as strings)
    df.rename(columns={
        "ex_dividend_date": "ex_dividend_date",
        "pay_date": "pay_date",
        "cash_amount": "cash_amount",
    }, inplace=True)

    df["ex_dividend_date"] = pd.to_datetime(df["ex_dividend_date"])
    if "pay_date" in df.columns:
        df["pay_date"] = pd.to_datetime(df["pay_date"])
    df = df.sort_values("ex_dividend_date")
    # print(df)
    return df


def get_splits(ticker: str, limit: int = 100) -> pd.DataFrame:
    """
    Fetch stock split history from Massive.
    """
    try:
        data = client.list_splits(ticker=ticker, limit=limit)
    except AttributeError:
        resp = client.get(f"/v2/splits/{ticker}", params={"limit": limit})
        data = resp.json()
        if isinstance(data, dict):
            data = data.get("results", [])

    df = pd.DataFrame(data)
    if df.empty:
        return df

    df.rename(columns={
        "execution_date": "execution_date",
        "split_from": "split_from",
        "split_to": "split_to",
    }, inplace=True)

    df["execution_date"] = pd.to_datetime(df["execution_date"])
    df = df.sort_values("execution_date")
    return df

# ### 3. Analysis logic
def check_adjustment(ticker: str) -> pd.DataFrame:
    """
    For a given ticker, examine each split and compare the last pre‑split dividend
    with the first post‑split dividend.
    """
    divs = get_dividends(ticker)
    splits = get_splits(ticker)

    if splits.empty:
        print(f"No splits found for {ticker}")
        return pd.DataFrame()

    results = []
    for _, split in splits.iterrows():
        split_date = split["execution_date"]
        from_shares = int(split["split_from"])
        to_shares = int(split["split_to"])
        factor = to_shares / from_shares   # >1 for a forward split (e.g. 2:1 → 2)

        pre_divs = divs[divs["ex_dividend_date"] < split_date].tail(1)
        # print(pre_divs)
        post_divs = divs[divs["ex_dividend_date"] >= split_date].head(1)
        # print(post_divs)

        if pre_divs.empty or post_divs.empty:
            continue

        pre_amt = pre_divs.iloc[0]["cash_amount"]
        post_amt = post_divs.iloc[0]["cash_amount"]
        if post_amt == 0:
            continue

        ratio = pre_amt / post_amt

        results.append({
            "split_date": split_date.date(),
            "from -> to": f"{from_shares} -> {to_shares}",
            "factor": factor,
            "pre_div_date": pre_divs.iloc[0]["ex_dividend_date"].date(),
            "pre_amount": pre_amt,
            "post_div_date": post_divs.iloc[0]["ex_dividend_date"].date(),
            "post_amount": post_amt,
            "ratio": ratio,
            "expected_if_unadjusted": factor,
            "expected_if_adjusted": 1.0,
            "likely_adjusted": abs(ratio - 1.0) < abs(ratio - factor),
        })

    return pd.DataFrame(results)

In [3]:
# ### 4. Run on a few classic split/dividend stocks

# Why ratio ≈ split factor means the data is NOT adjusted
# The pre_amount is the dividend as it was originally paid (e.g., $0.82 per pre‑split share).
# The post_amount is the dividend after the split, reflecting the new per‑share reality (e.g., $0.205).
# If the data were split‑adjusted, the pre‑split dividend would have been restated to $0.205 in the database (so that all dividends are expressed on the same per‑share basis). 
# Then pre/post would be 1.0, not 4.0.
# Since you see pre/post == 4.0, the pre‑split amount has not been scaled down – it’s still the original, unadjusted value.
# Every one of your tickers shows that Massive/Polygon dividends are NOT adjusted for splits. The likely_adjusted column is correctly False.

tickers_to_test = ["WMT", "AAPL", "AVGO", "FAST", "TSCO"]

for tkr in tickers_to_test:
    print(f"\n--- {tkr} ---")
    res = check_adjustment(tkr)
    if not res.empty:
        display(res[[
            "split_date", "from -> to", "pre_amount", "post_amount",
            "ratio", "expected_if_unadjusted", "expected_if_adjusted", "likely_adjusted"
        ]])


--- WMT ---


,split_date,from -> to,pre_amount,post_amount,ratio,expected_if_unadjusted,expected_if_adjusted,likely_adjusted
0,2024-02-26,1 -> 3,0.57,0.2075,2.746988,3.0,1.0,False



--- AAPL ---


,split_date,from -> to,pre_amount,post_amount,ratio,expected_if_unadjusted,expected_if_adjusted,likely_adjusted
0,2014-06-09,1 -> 7,3.29,0.470,7.0,7.0,1.0,False
1,2020-08-31,1 -> 4,0.82,0.205,4.0,4.0,1.0,False



--- AVGO ---


,split_date,from -> to,pre_amount,post_amount,ratio,expected_if_unadjusted,expected_if_adjusted,likely_adjusted
0,2024-07-15,1 -> 10,5.25,0.53,9.90566,10.0,1.0,False



--- FAST ---


,split_date,from -> to,pre_amount,post_amount,ratio,expected_if_unadjusted,expected_if_adjusted,likely_adjusted
0,2005-11-14,1 -> 2,0.31,0.20,1.550000,2.0,1.0,False
1,2011-05-23,1 -> 2,0.26,0.13,2.000000,2.0,1.0,False
2,2019-05-23,1 -> 2,0.43,0.22,1.954545,2.0,1.0,False
3,2025-05-22,1 -> 2,0.44,0.22,2.000000,2.0,1.0,False



--- TSCO ---


,split_date,from -> to,pre_amount,post_amount,ratio,expected_if_unadjusted,expected_if_adjusted,likely_adjusted
0,2010-09-03,1 -> 2,0.14,0.07,2.000000,2.0,1.0,False
1,2013-09-27,1 -> 2,0.26,0.13,2.000000,2.0,1.0,False
2,2024-12-20,1 -> 5,1.10,0.23,4.782609,5.0,1.0,False


# Volume data IS adjusted for splits by Polygon! That's great. Verified below.
## It's crucial to adjust Volume for Splits!


### Verifying whether Dividends are adjusted for Split already or not.
If volume isn’t split‑adjusted, a backtest that splits shares will see artificial volume jumps that can distort liquidity filters, VWAP calculations, and volume‑based signals.

The logic to verify is similar:
- For a forward split (e.g., 2:1), raw volume typically doubles because there are twice as many shares.
- If volume is adjusted in the database, pre‑split volumes are multiplied by the split factor, so pre_volume / post_volume ≈ 1.
- If not adjusted, post_volume / pre_volume ≈ split factor (or equivalently pre_volume / post_volume ≈ 1/factor).

The below fetches daily bars around split dates, calculates average volumes before and after, and flags the adjustment pattern.

* Ratio ≈ split factor (volume is **NOT** adjusted (raw))
* Ratio ≈ 1.0 (volume **IS** adjusted (restated to post‑split shares))



In [6]:
# ### Fetching daily bars for a ticker around a date range
def get_daily_bars(ticker: str, from_date: str, to_date: str) -> pd.DataFrame:
    """
    Fetch 1‑day aggregates for a given ticker between from_date and to_date (inclusive).
    """
    data = client.get_aggs(
        ticker=ticker,
        multiplier=1,
        timespan="day",
        from_=from_date,
        to=to_date,
        limit=5000
    )
    if isinstance(data, list):
        df = pd.DataFrame(data)
    else:
        df = pd.DataFrame(data.get("results", []))

    if df.empty:
        return df

    # Convert timestamp (Unix ms) to datetime
    if "timestamp" in df.columns:
        df["date"] = pd.to_datetime(df["timestamp"], unit="ms")
    elif "t" in df.columns:  # fallback
        df["date"] = pd.to_datetime(df["t"], unit="ms")
    else:
        raise KeyError("No timestamp column found in aggregates")

    df = df.sort_values("date")
    return df

# ### Split‑volume adjustment check

def check_volume_adjustment(ticker: str, window_days: int = 10) -> pd.DataFrame:
    """
    For each split, compute the ratio of average daily volume in the `window_days`
    after the split to the average daily volume before the split.
    The split date itself is excluded.
    """
    # Reuse the existing split‑fetching (adjust mapping as before)
    SPLITS_COLUMN_MAP = {
        "execution_date": "execution_date",
        "split_from": "split_from",
        "split_to": "split_to",
    }
    # Fetch splits
    splits = get_splits(ticker)  # using your earlier get_splits function
    if splits.empty:
        print(f"No splits for {ticker}")
        return pd.DataFrame()

    # We'll need a wide enough date range to cover all splits + window
    min_date = (splits["execution_date"].min() - timedelta(days=window_days+5)).strftime("%Y-%m-%d")
    max_date = (splits["execution_date"].max() + timedelta(days=window_days+5)).strftime("%Y-%m-%d")
    bars = get_daily_bars(ticker, min_date, max_date)
    if bars.empty or "volume" not in bars.columns:
        print(f"No volume data for {ticker}")
        return pd.DataFrame()

    results = []
    for _, split in splits.iterrows():
        split_date = split["execution_date"]
        from_shares = int(split["split_from"])
        to_shares = int(split["split_to"])
        factor = to_shares / from_shares   # e.g., 2.0 for a 2:1 split

        # Pre‑split window: [split_date - window_days, split_date - 1]
        pre_start = split_date - timedelta(days=window_days)
        pre_end = split_date - timedelta(days=1)
        pre_mask = (bars["date"] >= pre_start) & (bars["date"] <= pre_end)
        pre_vol = bars.loc[pre_mask, "volume"].mean()

        # Post‑split window: [split_date + 1, split_date + window_days]
        post_start = split_date + timedelta(days=1)
        post_end = split_date + timedelta(days=window_days)
        post_mask = (bars["date"] >= post_start) & (bars["date"] <= post_end)
        post_vol = bars.loc[post_mask, "volume"].mean()

        if pd.isna(pre_vol) or pd.isna(post_vol) or pre_vol == 0:
            continue

        ratio = post_vol / pre_vol   # if unadjusted, this ≈ factor

        results.append({
            "split_date": split_date.date(),
            "from -> to": f"{from_shares} -> {to_shares}",
            "factor": factor,
            "pre_avg_vol": pre_vol,
            "post_avg_vol": post_vol,
            "ratio": ratio,
            "expected_if_unadjusted": factor,
            "expected_if_adjusted": 1.0,
            "likely_adjusted": abs(ratio - 1.0) < abs(ratio - factor),
        })

    return pd.DataFrame(results)

In [7]:
# ### Run the verification on a few stocks
tickers_to_test = ["WMT", "AAPL", "AVGO", "FAST", "TSCO"]

for tkr in tickers_to_test:
    print(f"\n--- {tkr} ---")
    res = check_volume_adjustment(tkr)
    if not res.empty:
        display(res[[
            "split_date", "from -> to", "pre_avg_vol", "post_avg_vol",
            "ratio", "expected_if_unadjusted", "expected_if_adjusted", "likely_adjusted"
        ]])


--- WMT ---


,split_date,from -> to,pre_avg_vol,post_avg_vol,ratio,expected_if_unadjusted,expected_if_adjusted,likely_adjusted
0,2024-02-26,1 -> 3,47718477.6,1.850014e+07,0.387693,3.0,1.0,True



--- AAPL ---


,split_date,from -> to,pre_avg_vol,post_avg_vol,ratio,expected_if_unadjusted,expected_if_adjusted,likely_adjusted
0,2020-08-31,1 -> 4,2.333524e+08,2.250816e+08,0.964557,4.0,1.0,True



--- AVGO ---


,split_date,from -> to,pre_avg_vol,post_avg_vol,ratio,expected_if_unadjusted,expected_if_adjusted,likely_adjusted
0,2024-07-15,1 -> 10,4.036567e+07,3.100014e+07,0.767983,10.0,1.0,True



--- FAST ---


,split_date,from -> to,pre_avg_vol,post_avg_vol,ratio,expected_if_unadjusted,expected_if_adjusted,likely_adjusted
0,2019-05-23,1 -> 2,7.979770e+06,11297520.8,1.415770,2.0,1.0,True
1,2025-05-22,1 -> 2,5.631806e+06,4564195.8,0.810432,2.0,1.0,True



--- TSCO ---


,split_date,from -> to,pre_avg_vol,post_avg_vol,ratio,expected_if_unadjusted,expected_if_adjusted,likely_adjusted
0,2024-12-20,1 -> 5,4.924151e+06,3558143.75,0.72259,5.0,1.0,True


# Stock Dividends must be applied too! They are in 07_process.ipynb
Stock Dividends affect the integrity of your price action. Ignoring them corrupts your technical indicators, volatility calculations, and risk-management logic.

#### **Stock dividend adjustment — critical, non-negotiable**


Stock dividends cause a discrete price step on ex-date exactly like a split does. A 10% stock dividend drops the price ~9.1% overnight with no economic content. If you don't adjust:

Moving averages develop a phantom kink
Percentage-change calculations are distorted for the lookback window spanning the event
Patterns (ORB ranges, EP gap-up signals, relative volume) get corrupted
Breakout thresholds are miscalibrated

The effect is indistinguishable from a small split and must be treated identically. Not adjusting for stock dividends gives you spurious signals, inflated apparent volatility, and fake catalogue entries for price patterns. This is a data integrity issue, not a modelling choice.


# Getting Split and Dividend Adjustments for ALL tickers

In [2]:
# Loop through all tickers
tickers_v2 = get_tickers(v=2)
# To keep track of which dates the adjustments folder has, we save the end date to _end_date.txt.

# Updates only
# Use a different START_DATE if we only want to update
if not CLEAN_DOWNLOAD:
    if os.path.isfile(POLYGON_DATA_PATH + "raw/adjustments/_end_date.txt"):
        with open(POLYGON_DATA_PATH + "raw/adjustments/_end_date.txt") as f:
            end_date_of_data = next(f).strip()
            START_DATE = first_trading_date_after_equal(date.fromisoformat(end_date_of_data) + timedelta(days=1))
    else:
        raise Exception('There is no _end_date.txt file!')
        
for index, row in tickers_v2.iterrows():
    # if index <= 5739:
    #     continue
    ticker = row["ticker"]
    end_date = row["end_date"]

    # WRONG! Below is wrong if we are downloading data for backtesting, cos for backtesting we need to apply adjustments to 
    # even tickers that are currently delisted so we can backtest on these delisted tickers  too cos 
    # at some point historically they were active.
    # Tickers that do not need downloading/updating. This happens when the ticker is delisted before START_DATE.
    # if end_date < START_DATE:
    #     continue

    # Get data
    try:
        splits = pd.DataFrame(client.list_splits(ticker=ticker, execution_date_gte=START_DATE, execution_date_lte=END_DATE))
        dividends = pd.DataFrame(client.list_dividends(ticker=ticker, ex_dividend_date_gte=START_DATE, ex_dividend_date_lte =END_DATE))
    except Exception as e:
        print(repr(e))
        continue

    # Get correct format
    if not dividends.empty:
        dividends = dividends.rename(columns={'ex_dividend_date': 'date', 'dividend_type':'subtype', 'cash_amount': 'amount'})
        dividends['type'] = 'DIV'
        dividends = dividends[['date', 'type', 'subtype', 'amount']]

    if not splits.empty:
        splits = splits.rename(columns={'execution_date': 'date'})
        splits['type'] = 'SPLIT'
        splits['subtype'] = np.where(splits['split_from'] > splits['split_to'], 'R', 'N')
        splits['amount'] = splits['split_from'] / splits['split_to']
        splits = splits[['date', 'type', 'subtype', 'amount']]
    
    # Skip loop if no data
    if splits.empty and dividends.empty:
        continue

    # Merge dividends and splits
    adjustments = pd.concat([dividends, splits])
    adjustments = adjustments.sort_values(by='date').reset_index(drop=True)
    adjustments['date'] = pd.to_datetime(adjustments['date']).dt.date

    if adjustments.isnull().values.any():
        null_data = tickers_v3[tickers_v3[["ticker", "name", "active", "type", "start_date", "last_updated_utc"]].isnull().any(axis=1)]
        raise Exception(f"There are missing values for {ticker} at index {index}.")

    # Save or update
    path = POLYGON_DATA_PATH + f'raw/adjustments/{ticker}.csv'

    # If file exists, add new adjustments. This happens if a ticker is recycled or we are updating and the stock already has a history of adjustments.
    # Dev: No need to overthink it. Even for recycled tickers, adjustments obviously apply to the date in question and hence, 
    # the adjustment matches whatever recycled ticker is active at that time.
    if os.path.isfile(path):
        old_adjustments = pd.read_csv(
            path,
            parse_dates=True,
        )
        all_adjustments = pd.concat([old_adjustments, adjustments])
        all_adjustments['date'] = pd.to_datetime(all_adjustments['date']).dt.date
        all_adjustments = all_adjustments.sort_values(by='date').reset_index(drop=True)
        
        # Sometimes the adjustments are duplicated because multiple tickers or for no reason.
        if not all_adjustments[all_adjustments['type'] == 'DIV'].empty:
            dividends = all_adjustments[all_adjustments['type'] == 'DIV']
            indices_to_remove = dividends[dividends[['date', 'type']].duplicated(keep='last')].index
            all_adjustments = all_adjustments.drop(index=indices_to_remove)
        if not all_adjustments[all_adjustments['type'] == 'SPLIT'].empty:
            splits = all_adjustments[all_adjustments['type'] == 'SPLIT']
            indices_to_remove = splits[splits[['date', 'type']].duplicated(keep='last')].index
            all_adjustments = all_adjustments.drop(index=indices_to_remove)
            
        all_adjustments.to_csv(path, index=False)
    else:
        # Sometimes the adjustments are duplicated because multiple tickers or for no reason.
        if not adjustments[adjustments['type'] == 'DIV'].empty:
            dividends = adjustments[adjustments['type'] == 'DIV']
            indices_to_remove = dividends[dividends[['date', 'type']].duplicated(keep='last')].index
            adjustments = adjustments.drop(index=indices_to_remove)
        if not adjustments[adjustments['type'] == 'SPLIT'].empty:
            splits = adjustments[adjustments['type'] == 'SPLIT']
            indices_to_remove = splits[splits[['date', 'type']].duplicated(keep='last')].index
            adjustments = adjustments.drop(index=indices_to_remove)
        adjustments.to_csv(path, index=False)

# Set the END_DATE to end_date.txt
with open(POLYGON_DATA_PATH + "raw/adjustments/_end_date.txt", 'w') as f:
    f.write(f'{END_DATE}\n')

In [3]:
tickers_v2[tickers_v2['ticker'] == 'ESCA']

,ID,ticker,name,active,start_date,end_date,type,cik,composite_figi
3447,ESCA-2016-06-06,ESCA,Escalade Inc,True,2016-06-06,2026-06-03,CS,33488.0,BBG000BJ0H70


Some examples.

In [5]:
pd.read_csv(POLYGON_DATA_PATH + "raw/adjustments/MFA.csv", index_col="date").tail(7)

,type,subtype,amount
date,,,
2024-09-27,DIV,CD,0.35
2024-12-31,DIV,CD,0.35
2025-03-31,DIV,CD,0.36
2025-06-30,DIV,CD,0.36
2025-09-30,DIV,CD,0.36
2025-12-31,DIV,CD,0.36
2026-03-31,DIV,CD,0.36


In [6]:
pd.read_csv(POLYGON_DATA_PATH + "raw/adjustments/TSLA.csv", index_col="date")

,type,subtype,amount
date,,,
2020-08-31,SPLIT,N,0.200000
2022-08-25,SPLIT,N,0.333333


### Updates
Simply rerun the file with the correct <code>END_DATE</code> and set <code>CLEAN_DOWNLOAD</code> to False. The code is already structured in such a way that only the necessary is downloaded. Also, dividends have to be split adjusted. The code is below.